Данные:
Фондовые индексы: 
- CША: https://uk.investing.com/indices/us-30-historical-data (DJI)
- Великобритания: https://www.wsj.com/market-data/quotes/index/UK/UKX/historical-prices (FTSE 100)
- Германия: https://stooq.com/q/d/?f=19920101&t=20000425&s=%5Edax&c=0 (DAX)


Доходности долгосрочных облигаций:
- США: https://fred.stlouisfed.org/series/DGS10 (DGS10)
- Великобритания: https://www.bankofengland.co.uk/statistics/yield-curves (uk_10y_full)
- Германия: 

In [1]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

#### Функция для обаботки csv данных у которых значения в строковом виде

In [3]:
def process_string_csv_data(csv_path):
    out_df = pd.read_csv(
        csv_path,
        header=0,
        names=["Date", "Price", "Open", "High", "Low", "Vol", "Change_%"],
    )

    out_df["Date"] = pd.to_datetime(out_df["Date"], format="%d/%m/%Y")

    for col in out_df.columns[1:]:
        out_df[col] = (
            out_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)  
            .str.replace("%", "", regex=False)  
            .str.replace("M", "", regex=False)  
            .str.strip()
        )
        out_df[col] = pd.to_numeric(out_df[col], errors="coerce")
    return out_df


In [4]:
dji = process_string_csv_data("dji.csv")
print(dji.info())
print('\n-------------------\n')
print(dji.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2681 entries, 0 to 2680
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      2681 non-null   datetime64[us]
 1   Price     2681 non-null   float64       
 2   Open      2681 non-null   float64       
 3   High      2681 non-null   float64       
 4   Low       2681 non-null   float64       
 5   Vol       2101 non-null   float64       
 6   Change_%  2681 non-null   float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 146.7 KB
None

-------------------

                             Date         Price          Open          High  \
count                        2681   2681.000000   2681.000000   2681.000000   
mean   1996-06-19 08:25:57.627750   6420.434946   6417.801063   6449.283021   
min           1992-01-01 00:00:00   3136.580000   3149.010000   3168.830000   
25%           1994-07-22 00:00:00   3801.460000   3805.940000   3813.170000   
50%     

In [5]:
dgs = pd.read_csv('DGS10.csv')
dgs.head()

,observation_date,DGS10
0,1992-01-02,6.78
1,1992-01-03,6.85
2,1992-01-06,6.82
3,1992-01-07,6.76
4,1992-01-08,6.77


In [6]:
dgs["observation_date"] = pd.to_datetime(dgs["observation_date"], format="%Y-%m-%d")
dgs = dgs.rename(columns={"observation_date":"Date"})
dgs.head()

,Date,DGS10
0,1992-01-02,6.78
1,1992-01-03,6.85
2,1992-01-06,6.82
3,1992-01-07,6.76
4,1992-01-08,6.77


## Теперь облигации

In [7]:
print(dgs.info())
print(dgs.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2169 entries, 0 to 2168
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2169 non-null   datetime64[us]
 1   DGS10   2082 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 34.0 KB
None
                      Date        DGS10
count                 2169  2082.000000
mean   1996-02-28 00:00:00     6.283079
min    1992-01-02 00:00:00     4.160000
25%    1994-01-31 00:00:00     5.770000
50%    1996-02-28 00:00:00     6.250000
75%    1998-03-27 00:00:00     6.790000
max    2000-04-25 00:00:00     8.050000
std                    NaN     0.753005


### Далее есть некоторые проблемы с доступом к данным по фьючерсам (в открытых источниках нет данных по 1992-2000 годам, а к закрытым у нас нет доступа)
Поэтому есть альтернативный способ: вместо дневного изменения цены фьючерса на облигацию (как делают авторы) использовать приближение/упрощение - рассчитать изменение цены на облигацию через изменение доходности и модифицированную дюрацию

$$\frac{\Delta P}{P} \approx -D_{mod} \cdot \Delta i$$

Где $D_{mod}$ - модифицированная дюрация, $\Delta i$ - изменение доходности, а $\frac{\Delta P}{P}$ - изменение рыночной цены на облигацию

Изменение доходности $\Delta i$ мы имеем в DGS10 (и аналогах для других рынков), дюрацию же можно взять равной $7.5$ для упрощения

Берем такое значение по следующим причинам:
Для 10-ти летней купонной облигации с доходностью к погашению примерно равной 5.5%-6% дюрация Маколея примерна равна 8 лет

Тогда модифицированную дюрацию можно посчитать как: $D_{mod} = \frac{D_{mac}}{1 + \frac{y}{m}} $, где $y \approx 0.06$ (средняя ставка доходности 10-летних US Treasuries в 1990-х годах), $m = 2$ (полугодовые купонные выплаты)

При таком упрощении получаем $D_{mod} \approx 7.5$ 

Сначала объединяем по дате фондовый индекс и 

In [8]:
dji_prices = dji[["Date", "Price"]]
dji_prices.head()

,Date,Price
0,2000-04-25,11124.83
1,2000-04-24,10906.10
2,2000-04-23,10844.06
3,2000-04-22,10844.06
4,2000-04-21,10844.06


In [9]:
df_us = pd.merge(dji_prices, dgs, on="Date", how="inner")
df_us.head()

,Date,Price,DGS10
0,2000-04-25,11124.83,6.14
1,2000-04-24,10906.10,6.00
2,2000-04-21,10844.06,NaN
3,2000-04-20,10844.06,5.99
4,2000-04-19,10674.97,5.99


Теперь посмотрим NaNы

In [10]:
def check_nans(df):
    print("Количество пропусков:")
    print(df.isna().sum())

    print("\nПроцент пропусков:")
    print((df.isna().mean() * 100).round(2))

    total_nans = df.isna().sum().sum()
    print(f"\nВсего NaN в таблице: {total_nans}")

check_nans(df_us)

Количество пропусков:
Date      0
Price     0
DGS10    87
dtype: int64

Процент пропусков:
Date     0.00
Price    0.00
DGS10    4.01
dtype: float64

Всего NaN в таблице: 87


4 процента нанов, пропуски можно заполнить предыдущими значениями

In [11]:
df_us_interp = df_us.ffill()

In [12]:
check_nans(df_us_interp)

Количество пропусков:
Date     0
Price    0
DGS10    0
dtype: int64

Процент пропусков:
Date     0.0
Price    0.0
DGS10    0.0
dtype: float64

Всего NaN в таблице: 0


Теперь считаем s и i, Y через дюрацию и ER (избыт. доходность) как разность s и i

In [13]:
df_us_interp = df_us_interp.sort_values("Date").reset_index(drop=True)

In [14]:
df_us_interp["s_US"] = 100 * (np.log(df_us_interp["Price"]) - np.log(df_us_interp["Price"].shift(1)))
df_us_interp["i_US"] = df_us_interp["DGS10"] / 365.0
df_us_interp["delta_i_US"] = df_us_interp["DGS10"] - df_us_interp["DGS10"].shift(1)
df_us_interp["Y_US"] = -7.5 * df_us_interp["delta_i_US"]
df_us_interp["ER_US"] = df_us_interp["s_US"] - df_us_interp["i_US"]

df_us_interp.head(10)


,Date,Price,DGS10,s_US,i_US,delta_i_US,Y_US,ER_US
0,1992-01-02,3172.41,6.78,NaN,0.018575,NaN,NaN,NaN
1,1992-01-03,3201.47,6.85,0.911853,0.018767,0.07,-0.525,0.893086
2,1992-01-06,3200.13,6.82,-0.041865,0.018685,-0.03,0.225,-0.060549
3,1992-01-07,3204.83,6.76,0.146761,0.018521,-0.06,0.450,0.128241
4,1992-01-08,3203.93,6.77,-0.028087,0.018548,0.01,-0.075,-0.046635
5,1992-01-09,3209.53,6.79,0.174633,0.018603,0.02,-0.150,0.156030
6,1992-01-10,3199.46,6.85,-0.314246,0.018767,0.06,-0.450,-0.333013
7,1992-01-13,3185.60,6.92,-0.434139,0.018959,0.07,-0.525,-0.453098
8,1992-01-14,3246.20,7.03,1.884443,0.019260,0.11,-0.825,1.865182
9,1992-01-15,3258.50,7.05,0.378189,0.019315,0.02,-0.150,0.358873


In [15]:
check_nans(df_us_interp)

Количество пропусков:
Date          0
Price         0
DGS10         0
s_US          1
i_US          0
delta_i_US    1
Y_US          1
ER_US         1
dtype: int64

Процент пропусков:
Date          0.00
Price         0.00
DGS10         0.00
s_US          0.05
i_US          0.00
delta_i_US    0.05
Y_US          0.05
ER_US         0.05
dtype: float64

Всего NaN в таблице: 4


In [16]:
df_us_interp = df_us_interp.dropna()

In [17]:
check_nans(df_us_interp)

Количество пропусков:
Date          0
Price         0
DGS10         0
s_US          0
i_US          0
delta_i_US    0
Y_US          0
ER_US         0
dtype: int64

Процент пропусков:
Date          0.0
Price         0.0
DGS10         0.0
s_US          0.0
i_US          0.0
delta_i_US    0.0
Y_US          0.0
ER_US         0.0
dtype: float64

Всего NaN в таблице: 0


In [18]:
df_us_interp.describe()

,Date,Price,DGS10,s_US,i_US,delta_i_US,Y_US,ER_US
count,2168,2168.000000,2168.000000,2168.000000,2168.000000,2168.000000,2168.000000,2168.000000
mean,1996-02-28 16:48:15.940959,6110.562394,6.282265,0.057873,0.017212,-0.000295,0.002214,0.040661
min,1992-01-03 00:00:00,3136.580000,4.160000,-7.454915,0.011397,-0.230000,-2.925000,-7.471106
25%,1994-01-31 18:00:00,3699.072500,5.770000,-0.373440,0.015808,-0.030000,-0.225000,-0.390833
50%,1996-02-28 12:00:00,5474.600000,6.240000,0.031835,0.017096,0.000000,-0.000000,0.013994
75%,1998-03-27 18:00:00,8255.992500,6.790000,0.535662,0.018603,0.030000,0.225000,0.520018
max,2000-04-25 00:00:00,11722.980000,8.050000,4.860535,0.022055,0.390000,1.725000,4.846727
std,NaN,2646.107577,0.755120,0.899431,0.002069,0.057842,0.433812,0.899493


Вообще нормально, все похоже на таблицу только вот mean для ER у авторов 0.032 а у нас ~0.041, но остальные параметры сходятся довольно точно

Дальше оцениваем OLS модель (но перед этим надо еще добавить все предыдущие статистики)

In [19]:
import statsmodels.api as sm
X = df_us_interp["Y_US"]
X = sm.add_constant(X)
Y = df_us_interp["ER_US"]

model = sm.OLS(Y, X).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  ER_US   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     52.67
Date:                Sat, 19 Sep 2026   Prob (F-statistic):           5.48e-13
Time:                        13:05:52   Log-Likelihood:                -2820.1
No. Observations:                2168   AIC:                             5644.
Df Residuals:                    2166   BIC:                             5656.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0400      0.019      2.093      0.036       0.003       0.077
Y_US           0.3195      0.044      7.257      0.000       0.233       0.406
==============================================================================
Omnibus:                      461.926   Durbin-Watson:                   1.964
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             5525.082
Skew:                          -0.650   Prob(JB):                         0.00
Kurtosis:                      10.712   Cond. No.                         2.31
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [20]:
df_us_interp["EX_US"] = model.resid
df_us_interp[["ER_US", "Y_US", "EX_US"]].head(10)

,ER_US,Y_US,EX_US
1,0.893086,-0.525,1.020848
2,-0.060549,0.225,-0.172382
3,0.128241,0.450,-0.055470
4,-0.046635,-0.075,-0.062629
5,0.156030,-0.150,0.163995
6,-0.333013,-0.450,-0.229211
7,-0.453098,-0.525,-0.325336
8,1.865182,-0.825,2.088783
9,0.358873,-0.150,0.366838
10,-0.294578,-0.600,-0.142857


In [21]:
df_us_interp[["ER_US", "Y_US", "EX_US"]].describe()

,ER_US,Y_US,EX_US
count,2168.000000,2168.000000,2.168000e+03
mean,0.040661,0.002214,-2.949670e-17
std,0.899493,0.433812,8.887532e-01
min,-7.471106,-2.925000,-7.750655e+00
25%,-0.390833,-0.225000,-4.070509e-01
50%,0.013994,-0.000000,-6.533821e-03
75%,0.520018,0.225000,4.629875e-01
max,4.846727,1.725000,4.806773e+00


Повторяем первую таблицу

In [23]:
import numpy as np
from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

def calculate_table1_stats(series):
    s = series.dropna()
    
    stats = {
        "Mean": s.mean(),
        "Median": s.median(),
        "Std. Dev.": s.std(),
        "Skewness": skew(s),
        "Kurtosis": kurtosis(s, fisher=False) 
    }

    jb_stat, jb_pvalue = jarque_bera(s)
    stats["JB"] = jb_stat

    lb_test = acorr_ljungbox(s, lags=[1, 5], return_df=True)
    stats["LB(1)"] = lb_test['lb_stat'].iloc[0]
    stats["LB(5)"] = lb_test['lb_stat'].iloc[1]

    adf_result = adfuller(s, maxlag=10, autolag=None, result_object=False)
    stats["ADF(10)"] = adf_result[0]

    return pd.Series(stats)


table_1_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(df_us_interp["ER_US"]),
    "YUS": calculate_table1_stats(df_us_interp["Y_US"]),
    "DLDJIA": calculate_table1_stats(df_us_interp["s_US"]),
})

print(table_1_us.round(4))

                ERUS       YUS     DLDJIA
Mean          0.0407    0.0022     0.0579
Median        0.0140    0.0000     0.0318
Std. Dev.     0.8995    0.4338     0.8994
Skewness     -0.5398   -0.4121    -0.5424
Kurtosis      9.5187    5.6941     9.5241
JB         3943.8071  717.0110  3951.2128
LB(1)         0.7976   13.2808     0.7885
LB(5)        10.3395   30.3788    10.3680
ADF(10)     -14.5175  -13.7888   -14.5283


Теперь когда все протестировано можно слить с остальными данными